# MedAgent: 06. Agentic AI Clinical Reasoning & RAG Evaluation
## Multi-Agent Workflow, Knowledge Retrieval, and Explainable Report Generation

This notebook evaluates the Multi-Agent clinical decision support framework:
1. **Signal Analysis Agent**: Interprets physiological ECG/PPG findings and rhythm irregularities.
2. **Imaging Analysis Agent**: Interprets radiologic abnormalities and anatomical Grad-CAM regions.
3. **Evidence Agent (RAG)**: Retrieves medical literature and clinical practice guidelines.
4. **Risk Assessment Agent**: Computes composite risk score and flags contraindications.
5. **Report Generation Agent**: Formats comprehensive, clinician-facing decision-support reports.
6. **Supervisor Agent**: Validates safety guardrails and evidence grounding.

In [ ]:
import sys
from pathlib import Path
import json

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

### 1. Simulated Clinical Patient Case

In [ ]:
patient_case = {
    "patient_id": "PT-94021",
    "demographics": {
        "age": 68,
        "sex": "Male",
        "hypertension": True,
        "diabetes": True,
        "smoking_history": "Former (20 pack-years)"
    },
    "vitals": {
        "heart_rate_bpm": 96,
        "blood_pressure": "148/92 mmHg",
        "spo2_percent": 94
    },
    "ai_model_predictions": {
        "ecg_finding": "Ventricular Ectopic Beat (PVC)",
        "ecg_confidence": 0.91,
        "xray_finding": "Cardiomegaly with Mild Pleural Effusion",
        "xray_confidence": 0.88,
        "gradcam_region": "Lower left thoracic cavity and enlarged cardiac silhouette",
        "multimodal_risk_level": "High / Critical Risk",
        "multimodal_risk_score": 0.87
    }
}

print(json.dumps(patient_case, indent=2))

### 2. Multi-Agent Step-by-Step Reasoning Simulation

In [ ]:
def simulate_multi_agent_workflow(case):
    # 1. Signal Agent
    ecg_pred = case["ai_model_predictions"]["ecg_finding"]
    ecg_conf = case["ai_model_predictions"]["ecg_confidence"]
    signal_thought = (
        f"[Signal Agent]: Detected '{ecg_pred}' with {ecg_conf*100:.1f}% confidence. "
        f"The QRS complex demonstrates premature ventricular morphology with compensatory pause. "
        f"Elevated resting heart rate of {case['vitals']['heart_rate_bpm']} BPM noted."
    )
    
    # 2. Imaging Agent
    xray_pred = case["ai_model_predictions"]["xray_finding"]
    xray_conf = case["ai_model_predictions"]["xray_confidence"]
    gradcam_loc = case["ai_model_predictions"]["gradcam_region"]
    imaging_thought = (
        f"[Imaging Agent]: Radiograph analysis reveals '{xray_pred}' ({xray_conf*100:.1f}% confidence). "
        f"Grad-CAM attention focuses on: {gradcam_loc}."
    )
    
    # 3. Evidence Agent (RAG)
    evidence_retrieved = [
        "ACC/AHA Guidelines for Ventricular Arrhythmias (2022): Frequent PVCs in the presence of structural heart disease warrant echocardiographic evaluation.",
        "Fleischner Society Guidelines: Bilateral or dependent pleural effusion combined with cardiomegaly is highly indicative of congestive heart failure (CHF) decompensation."
    ]
    evidence_thought = (
        f"[Evidence Agent]: Retrieved 2 high-relevance clinical practice guidelines:\n" +
        "\n".join([f"  • {e}" for e in evidence_retrieved])
    )
    
    # 4. Risk Assessment Agent
    risk_level = case["ai_model_predictions"]["multimodal_risk_level"]
    risk_thought = (
        f"[Risk Assessment Agent]: Synthesizing multimodal findings. "
        f"Co-occurrence of PVC arrhythmia, cardiomegaly, hypertension, and borderline hypoxia (SpO2 {case['vitals']['spo2_percent']}%) "
        f"indicates elevated cardiovascular decompensation risk. Final Risk Tier: {risk_level}."
    )
    
    # 5. Report Generation Agent
    clinical_report = f"""
================================================================================
                    MEDAGENT CLINICAL DECISION SUPPORT REPORT
================================================================================
PATIENT ID: {case['patient_id']} | AGE: {case['demographics']['age']} | SEX: {case['demographics']['sex']}
DATE: 2026-09-29 | STATUS: PENDING CLINICIAN REVIEW
--------------------------------------------------------------------------------
1. PRIMARY FINDINGS:
   • ECG: {ecg_pred} (Confidence: {ecg_conf*100:.1f}%)
   • Imaging: {xray_pred} (Confidence: {xray_conf*100:.1f}%)
   • Key Localizations: {gradcam_loc}

2. MULTIMODAL RISK STRATIFICATION:
   • Risk Severity: {risk_level} (Calculated Index: {case['ai_model_predictions']['multimodal_risk_score']})
   • Risk Drivers: PVC burden combined with radiologic cardiomegaly and hypertension history.

3. SUPPORTING CLINICAL EVIDENCE:
   • ACC/AHA (2022): Frequent ventricular ectopy with cardiomegaly requires structural workup.
   • Fleischner Society: Effusion in setting of enlarged cardiac silhouette suggests heart failure.

4. RECOMMENDED CLINICAL ACTIONS:
   [ ] Perform 12-lead diagnostic ECG and comprehensive echocardiogram.
   [ ] Check serum electrolytes, BNP/NT-proBNP, and renal function.
   [ ] Evaluate fluid status and adjust antihypertensive/diuretic therapy as clinically indicated.
   [!] CAUTION: MedAgent is an AI clinical assistant. Final diagnosis rests with the attending physician.
================================================================================
"""
    return {
        "signal_thought": signal_thought,
        "imaging_thought": imaging_thought,
        "evidence_thought": evidence_thought,
        "risk_thought": risk_thought,
        "report": clinical_report
    }

agent_outputs = simulate_multi_agent_workflow(patient_case)
print(agent_outputs["signal_thought"])
print()
print(agent_outputs["imaging_thought"])
print()
print(agent_outputs["evidence_thought"])
print()
print(agent_outputs["risk_thought"])
print()
print(agent_outputs["report"])